
## Exploring Profit and loss statement

### Importing libraries

In [2]:
import pandas as pd
import requests
import re
import json

### Fetch HTML from Codal

In [3]:
url = "https://codal.ir/Reports/Decision.aspx?LetterSerial=7DuGSKY3REXqwohrFpHVcA%3d%3d&rt=0&let=6&ct=0&ft=-1&sheetId=1"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
html = response.text


In [9]:
pattern = re.compile(r"var datasource\s*=\s*({[\w\W]+?});")
match = pattern.search(html)

if match:
    json_str = match.group(1)
    data = json.loads(json_str)
else:
    data = None


In [11]:
if data:
    print("Top-level keys:", data.keys())
    print('Number of keys:',len(data.keys()))


Top-level keys: dict_keys(['title_Fa', 'title_En', 'subject', 'dsc', 'type', 'period', 'periodEndToDate', 'yearEndToDate', 'periodExtraDay', 'isConsolidated', 'tracingNo', 'kind', 'isAudited', 'auditState', 'registerDateTime', 'sentDateTime', 'publishDateTime', 'state', 'isForAuditing', 'sheets'])
Number of keys: 20


### Explore sheets Key

In [ ]:
# Explore sheets
sheets = data.get("sheets", [])

print(f"Number of sheets: {len(sheets)}")

# Preview first sheet
if sheets:
    sheet0 = sheets[0]
    print("Sheet keys:", sheet0.keys())
    print("Sheet 0 sample:")
    print(json.dumps(sheet0, indent=2, ensure_ascii=False))  # Pretty print with Persian chars


Number of sheets: 1
Sheet keys: dict_keys(['code', 'title_Fa', 'title_En', 'sequence', 'isDynamic', 'tables', 'aliasName', 'versionNo', 'sheetComponents'])
Sheet 0 sample:
{
  "code": 1,
  "title_Fa": "صورت سود و زیان",
  "title_En": "Income Statement",
  "sequence": 7,
  "isDynamic": true,
  "tables": [
    {
      "metaTableId": 3234,
      "title_En": "Income Statement",
      "title_Fa": "صورت سود و زيان",
      "sequence": 1,
      "sheetCode": 1,
      "code": 904,
      "description": null,
      "aliasName": "IncomeStatement",
      "versionNo": "4",
      "cells": [
        {
          "metaTableId": 3234,
          "metaTableCode": 904,
          "address": "A1",
          "formula": "",
          "validations": "",
          "financialConcept": null,
          "cellGroupName": "Header",
          "rowSpan": 2,
          "rowCode": 1,
          "rowSequence": 1,
          "colSpan": 1,
          "columnCode": 1,
          "columnSequence": 1,
          "cssClass": "",
       

In [ ]:
# Access the first sheet
sheet0 = sheets[0]

# Access the first table inside the sheet
table0 = sheet0["tables"][0]

# Now get the cells and turn them into a DataFrame
df = pd.DataFrame(table0["cells"])
df


In [19]:
import pandas as pd

# Suppose this is the "cells" list from the JSON
cells = table0["cells"]  # from sheet['tables'][0]

df = pd.DataFrame(cells)

# Step 1: Filter to Body rows only
df = df[df['cellGroupName'] == 'Body']

# Step 2: Keep only value-type rows (ignore formulas, hidden rows, etc.)
df = df[df['isVisible'] == True]

# Step 3: Separate label and value columns
labels = df[df['columnCode'] == 1][['rowCode', 'value']].rename(columns={'value': 'label'})
values = df[df['columnCode'] == 2][['rowCode', 'value']].rename(columns={'value': 'amount'})

# Step 4: Merge on rowCode
income_df = pd.merge(labels, values, on='rowCode', how='inner')

# Optional: Clean numbers
income_df['amount'] = pd.to_numeric(income_df['amount'].str.replace(',', '').str.replace('(', '-').str.replace(')', ''), errors='coerce')

# Final result
income_df


,rowCode,label,amount
0,49,عمليات در حال تداوم:,NaN
1,3,درآمدهاي عملياتي,2.763556e+09
2,4,بهاى تمام شده درآمدهاي عملياتي,-1.803741e+09
3,5,سود(زيان) ناخالص,9.598149e+08
4,6,هزينه‏ هاى فروش، ادارى و عمومى,-1.170736e+08
5,44,هزينه کاهش ارزش دريافتني‏ ها (هزينه استثنايي),0.000000e+00
6,7,ساير درآمدها,5.336709e+07
7,8,ساير هزينه‌ها,-8.213330e+05
8,9,سود(زيان) عملياتى,8.952871e+08
9,10,هزينه‏ هاى مالى,-1.309046e+08


In [24]:
with open("codal_full_pretty.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)


In [ ]:
from scraper.income_statement import extract_income_statement
from scraper.link_collector import get_announcement_links

links = get_announcement_links(page=1)  # already returns links with &sheetId=1

for url in links:
    df = extract_income_statement(url)
    if df is not None:
        print(df.head())
        # Optionally save to DB or CSV


In [26]:
df

,metaTableId,metaTableCode,address,formula,validations,financialConcept,cellGroupName,rowSpan,rowCode,rowSequence,...,isVisible,value,valueTypeName,dataTypeName,decimalPlace,periodEndToDate,yearEndToDate,conditionalIsUnique,removableCustomRow,isAudited
1,3234,904,A3,,,None,Body,1,49,3,...,True,عمليات در حال تداوم:,Fix,None,0,,,False,None,False
2,3234,904,A4,,,None,Body,1,3,4,...,True,درآمدهاي عملياتي,Fix,None,0,,,False,None,False
3,3234,904,A5,,,None,Body,1,4,5,...,True,بهاى تمام شده درآمدهاي عملياتي,Fix,None,0,,,False,None,False
4,3234,904,A6,,,None,Body,1,5,6,...,True,سود(زيان) ناخالص,Fix,None,0,,,False,None,False
5,3234,904,A7,,,None,Body,1,6,7,...,True,هزينه‏ هاى فروش، ادارى و عمومى,Fix,None,0,,,False,None,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,3234,904,F25,"IF(B25*D25>0,ROUND((B25-D25)*100/ABS(D25),0),""...",,None,Body,1,47,25,...,True,7,-1,None,0,,,False,None,False
172,3234,904,F26,"IF(B26*D26>0,ROUND((B26-D26)*100/ABS(D26),0),""...",,None,Body,1,48,26,...,True,0,-1,None,0,,,False,None,False
173,3234,904,F27,"IF(B27*D27>0,ROUND((B27-D27)*100/ABS(D27),0),""...",,None,Body,1,21,27,...,True,7,-1,None,0,,,False,None,False
174,3234,904,F28,"IF(B28*D28>0,ROUND((B28-D28)*100/ABS(D28),0),""...",,None,Body,1,41,28,...,True,8,-1,None,0,,,False,None,False


In [28]:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "fa-IR,fa;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://codal.ir/",
    "Connection": "keep-alive",
}

session = requests.Session()
session.headers.update(headers)

response = session.get("https://codal.ir/ReportList.aspx?search&LetterType=-1&AuditorRef=-1&PageNumber=2&Audited&NotAudited=false&IsNotAudited=false&Childs=false&Mains&Publisher=false&CompanyState=-1&ReportingType=-1&Category=1&CompanyType=-1&Consolidatable&NotConsolidatable")
print(response.status_code)
print(response.text[:500])


200
<!DOCTYPE html><html dir="rtl" lang="fa"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>دسترسی مسدود شد</title><style>:root{--f5-blue:#0061A8;--f5-red:#E41D3D}body{font-family:Tahoma,Arial,sans-serif;background-color:#f8f9fa;margin:0;padding:0;height:100vh;display:flex;justify-content:center;align-items:center}.block-container{background-color:#fff;padding:2.5rem;border-radius:8px;box-shadow:0 4px 20px rgba(0,0,0,.1);text-align:center;max-wi


In [4]:
import requests
from bs4 import BeautifulSoup

url = "https://codal.ir/Reports/InterimStatement.aspx?LetterSerial=0sdMovDOOObOOOU2nl7iZ7eh4N9g%3d%3d"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

In [8]:
# Try basic
table = soup.find_all("table")

# If needed, inspect in browser and try more specific:
# table = soup.find("table", {"class": "rpt"})

# Preview the table HTML
print(table[7])  # Just first 1000 chars to avoid overload


<table cellpadding="0" cellspacing="0" width="100%">
<tr>
<td>
</td>
<td align="right" style="font-weight: bold;">
<span id="ctl00_cphBody_ucLetterAuditingV2_lblAuditorName">موسسه حسابرسي هدف نوين نگر</span>
</td>
</tr>
<tr>
<td style="width: 40%;">
</td>
<td align="right" style="width: 60%;" valign="top">
<table cellpadding="0" cellspacing="0" style="width: 100%; padding: 0px; height: 100%;
    font-size: 8pt; font-family: tahoma; border-color: #808080; border-style: none;
    border-width: 0px; background-color: White;">
<tr style="width: 100%;" valign="top">
<td>
<table cellpadding="1" cellspacing="0" style="width: 100%; font-size: 8pt; font-family: tahoma;">
<tr style="height: 100%;" valign="top">
<td>
<div>
<table border="1" cellpadding="3" cellspacing="0" class="Grid" id="ctl00_cphBody_ucLetterAuditingV2_ucLetterSigner1_dgSigner" rules="rows" style="border-color:#E0E0E0;border-width:1px;border-style:Solid;width:100%;border-collapse:collapse;" tabindex="6">
<tr class="GridHeader">

In [ ]:
from bs4 import BeautifulSoup
import requests

url = "https://codal.ir/Reports/InterimStatement.aspx?LetterSerial=0sdMovDOOObOOOU2nl7iZ7eh4N9g%3d%3d"
url=url + '&sheetId=1'
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

# Step 1: Find the div with class 'table_wrapper'
wrapper_div = soup.find("div", class_="table_wrapper")
print(wrapper_div)
# Step 2: Get all tables inside that div
tables = wrapper_div.find_all("table")

# Step 3: Loop through the tables and print their rows
for i, table in enumerate(tables, start=1):
    print(f"\n===== TABLE {i} =====")
    for row in table.find_all("tr"):
        cols = row.find_all(["td", "th"])
        texts = [col.get_text(strip=True) for col in cols]
        print(texts)


<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">

<html xmlns="http://www.w3.org/1999/xhtml">
<head id="ctl00_Head1"><title>
	 صورت‌های مالی 12 ماهه منتهی به  1404/06/31 (حسابرسی شده)
</title><link href="../css/Main.css" rel="stylesheet" type="text/css"/><link href="../css/fepsProductV1.css" rel="stylesheet" type="text/css"/><link href="../assets/favicon/favicon.ico?v=RyQ5xz7ege" rel="shortcut icon"/>
<!--[if lt IE 7]>
	<link href="../css/IE6_new_style.css" type="text/css" rel="stylesheet" />
	<![endif]-->
<script language="javascript" src="../js/PKI.js" type="text/javascript"></script>
<script language="javascript" src="../js/Common.js" type="text/javascript"></script>
<script language="javascript" src="../js/jquery-1.8.3.min.js" type="text/javascript"></script>
<script language="javascript" type="text/javascript">
        document.addEventListener("DOMContentLoaded", function () {
            var priceNote = do

AttributeError: 'NoneType' object has no attribute 'find_all'

In [1]:
import pandas as pd
df=pd.read_csv(r'C:\Users\amirmahdi\Desktop\codal scraper\merged_income_statements.csv')

In [2]:
df

,rowCode,label,amount,company_name,source_url,amount as of تاریخ نامشخص
0,49.0,عمليات در حال تداوم:,NaN,قند تربت حيدريه,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
1,3.0,درآمدهاي عملياتي,16420163.0,قند تربت حيدريه,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
2,4.0,بهاى تمام شده درآمدهاي عملياتي,-15258131.0,قند تربت حيدريه,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
3,5.0,سود(زيان) ناخالص,1162032.0,قند تربت حيدريه,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
4,6.0,هزينه‏ هاى فروش، ادارى و عمومى,-423074.0,قند تربت حيدريه,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
...,...,...,...,...,...,...
361,32.0,ناشی از عملیات در حال تداوم,913.0,کشت و صنعت و دامپروري پگاه فارس,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
362,33.0,ناشی از عملیات متوقف شده,0.0,کشت و صنعت و دامپروري پگاه فارس,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
363,34.0,سود (زیان) پایه هر سهم,913.0,کشت و صنعت و دامپروري پگاه فارس,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
364,35.0,سود (زیان) خالص هر سهم– ریال,913.0,کشت و صنعت و دامپروري پگاه فارس,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN


In [1]:
import pandas as pd
df=pd.read_csv(r'C:\Users\amirmahdi\Desktop\codal scraper\output\companies_cleaned.csv')

In [6]:
df[df['company_id'].isnull()==True]

,ticker,name,company_id,publisher_status_id,company_type_id,industry_groups_id,company_nature_id,created_at,audited
3,پترونیک,مهندسی پترونیک صنعت,NaN,-1,3,-1,1000005,2025-10-29 09:11:47,False
293,ارتباطات پیوسته ایرانیان,ارتباطات پیوسته ایرانیان,NaN,-1,3,-1,1000005,2025-10-29 09:11:47,False
416,انرژی آسماری کیش,انرژی آسماری کیش,NaN,-1,3,-1,1000005,2025-10-29 09:11:47,False
2491,وجهاد,سرمایه گذاری صبا جهاد,NaN,3,2,-1,1000005,2025-10-29 09:11:47,False
3928,مادر تخصصی مدیریت منابع آب ایران,شرکت مادر تخصصی مدیریت منابع آب ایران,NaN,5,3,-1,1000005,2025-10-29 09:11:47,False
4189,مهندسی توسعه آب آسیا,مهندسی توسعه آب آسیا,NaN,-1,3,-1,1000005,2025-10-29 09:11:47,False
4503,هلد پترو,پترو فرهنگ,NaN,3,2,-1,1000002,2025-10-29 09:11:47,False
4527,هلد فیروزه,گروه توسعه مالی فیروزه,NaN,3,2,-1,1000002,2025-10-29 09:11:47,False
4530,هلد لیان,گروه پیشگامان اعتماد لیان ایرانیان,NaN,3,2,-1,1000002,2025-10-29 09:11:47,False
4542,هلد فولاد,گروه توسعه فراگیر فولاد خوزستان,NaN,3,2,-1,1000002,2025-10-29 09:11:47,False


In [48]:
df['industry_groups_id'].unique()

array([81, -1, 46, 40,  1, 72, 56, 77, 42, 66, 44, 73, 23, 13, 43, 33, 70,
       45, 27, 64, 94, 65, 57, 47, 29, 67, 31, 71, 26, 50, 34, 25, 11, 14,
       74, 58, 15, 60, 39, 38, 76, 55, 22, 21, 20, 28, 19, 62, 61, 63, 10,
       53, 54, 49, 12, 17, 32, 84, 51, 80], dtype=int64)

In [25]:
df.to_csv("output/companies_cleaned.csv", index=False, encoding='utf-8-sig')

In [31]:
df['company_type_id'][df['company_type_id']==5]=1

C:\Users\amirmahdi\AppData\Local\Temp\ipykernel_8484\2231215748.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['company_type_id'][df['company_type_id']==5]=1


In [36]:
df['company_type_id'].unique()

array([-1,  1,  0], dtype=int64)

In [45]:
df['industry_groups_id'][df['industry_groups_id']==91]=-1

C:\Users\amirmahdi\AppData\Local\Temp\ipykernel_8484\4085518796.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['industry_groups_id'][df['industry_groups_id']==91]=-1


In [49]:
df[df['industry_groups_id'] == -1]

,ticker,name,company_id,publisher_status_id,company_type_id,industry_groups_id,company_nature_id,created_at,audited
3,پترونیک,مهندسی پترونیک صنعت,NaN,-1,-1,-1,1000005,2025-10-29 09:11:47,False
6,مهندسی بهنیان جنوب,مهندسی بهنیان جنوب,751124.0,4,-1,-1,1000005,2025-10-29 09:11:47,False
43,آب و فاضلاب استان آذربایجان شرقی,آب و فاضلاب استان آذربایجان شرقی,751152.0,4,-1,-1,1000005,2025-10-29 09:11:47,False
50,آبفا,آب و فاضلاب استان خراسان جنوبی,751428.0,4,-1,-1,1000005,2025-10-29 09:11:47,False
51,آب و فاضلاب استان خراسان رضوی,آب و فاضلاب استان خراسان رضوی,751205.0,-1,-1,-1,1000005,2025-10-29 09:11:47,False
...,...,...,...,...,...,...,...,...,...
5267,کود آلی کرمانشاه,بازیافت مواد و تولید کود آلی کرمانشاه,751160.0,4,-1,-1,1000000,2025-10-29 09:11:47,False
5286,کیش آیمت آسیا,تاسیسات دریایی کیش آیمت آسیا,791296.0,-1,-1,-1,1000005,2025-10-29 09:11:47,False
5287,کیش بتون,کیش بتون جنوب,791232.0,4,1,-1,1000000,2025-10-29 09:11:47,False
5293,کیمیا فسفات,مجتمع صنعتی کیمیا فسفات پارسیان,791371.0,4,-1,-1,1000000,2025-10-29 09:11:47,False


In [50]:
df.describe()

,company_id,publisher_status_id,company_type_id,industry_groups_id,company_nature_id
count,5.283000e+03,5300.000000,5300.000000,5300.000000,5.300000e+03
mean,9.632933e+06,1.522830,-0.257925,55.052642,1.000003e+06
std,1.896684e+07,2.153316,0.913551,29.097205,2.083119e+00
min,0.000000e+00,-1.000000,-1.000000,-1.000000,1.000000e+06
25%,7.513415e+05,-1.000000,-1.000000,44.000000,1.000002e+06
50%,8.108050e+05,1.000000,-1.000000,65.000000,1.000005e+06
75%,8.122415e+05,3.000000,1.000000,81.000000,1.000005e+06
max,4.649906e+08,5.000000,1.000000,94.000000,1.000009e+06


In [60]:
df['industry_groups_id'][df['industry_groups_id']==-1]=None

C:\Users\amirmahdi\AppData\Local\Temp\ipykernel_8484\2976622510.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['industry_groups_id'][df['industry_groups_id']==-1]=None
C:\Users\amirmahdi\AppData\Local\Temp\ipykernel_8484\2976622510.p

In [67]:
df['industry_groups_id'].dtype

dtype('float64')

In [65]:
fk_cols = ['industry_groups_id', 'publisher_status_id', 'company_type_id', 'company_nature_id']

for col in fk_cols:
    df[col] = df[col].apply(lambda x: int(x) if pd.notnull(x) and x != -1 else None)


In [68]:
df['publisher_status_id'].dtype

dtype('float64')

In [73]:
df['publisher_status_id'][df['publisher_status_id']==2]

2781    2.0
2782    2.0
2783    2.0
2784    2.0
2785    2.0
       ... 
5081    2.0
5082    2.0
5083    2.0
5084    2.0
5085    2.0
Name: publisher_status_id, Length: 122, dtype: float64

In [75]:
df.dtype()

AttributeError: 'DataFrame' object has no attribute 'dtype'

In [3]:
# List of columns you want to convert
fk_cols = ['industry_groups_id', 'publisher_status_id', 'company_type_id', 'company_nature_id']

# Apply conversion: float to int, -1 or NaN to None
for col in fk_cols:
    df[col] = df[col].apply(lambda x: int(x) if pd.notnull(x) and x != -1 else None)

In [13]:
df

,ticker,name,company_id,publisher_status_id,company_type_id,industry_groups_id,company_nature_id,created_at,audited
0,آذرنگین,آذر نگین,811620.0,-1,-1,81.0,1000005,2025-10-29 09:11:47,False
1,بنیاد تعاون نیروی انتظامی جمهوری اسلامی ایران,بنیاد تعاون نیروی انتظامی جمهوری اسلامی ایران,811774.0,-1,-1,81.0,1000005,2025-10-29 09:11:47,False
2,بین المللی خدمات مسافرت هوایی ایرانگردی و جها...,بین المللی خدمات مسافرت هوایی ایرانگردی و جهان...,811602.0,-1,-1,81.0,1000005,2025-10-29 09:11:47,False
3,پترونیک,مهندسی پترونیک صنعت,NaN,-1,-1,NaN,1000005,2025-10-29 09:11:47,False
4,توس پیوند,توس پیوند,811130.0,-1,-1,81.0,1000005,2025-10-29 09:11:47,False
...,...,...,...,...,...,...,...,...,...
5295,کیمیاورزان مهر غرب,کیمیا ورزان مهر غرب,751383.0,4,1,NaN,1000000,2025-10-29 09:11:47,False
5296,کیمیای پارس خاور میانه,کیمیای پارس خاور میانه,812140.0,-1,-1,81.0,1000005,2025-10-29 09:11:47,False
5297,کیمیایی غرب گستر,صنایع کیمیای غرب گستر,811122.0,-1,-1,81.0,1000005,2025-10-29 09:11:47,False
5298,کیهان دریا ایمن قشم,کیهان دریا ایمن قشم,810884.0,-1,-1,81.0,1000000,2025-10-29 09:11:47,False


In [35]:
import pandas as pd
df=pd.read_csv(r'C:\Users\amirmahdi\Desktop\codal scraper\output\companies_cleaned.csv')

In [10]:
import numpy as np

fk_cols = ['industry_groups_id', 'publisher_status_id', 'company_type_id', 'company_nature_id']

def clean_foreign_key(val):
    try:
        val = float(val)  # Handle strings like "81.0"
        if np.isnan(val) or val == -1:
            return None
        return int(val)
    except:
        return None

for col in fk_cols:
    df[col] = df[col].apply(clean_foreign_key)

In [14]:
import numpy as np

def clean_industry_group(val):
    try:
        val = float(val)  # handles string like "81.0"
        if np.isnan(val) or val == -1:
            return None
        return int(val)
    except:
        return None

df['industry_groups_id'] = df['industry_groups_id'].apply(clean_industry_group)


In [15]:
print(df['industry_groups_id'].unique())
print(df['industry_groups_id'].dtype)


[81. nan 46. 40.  1. 72. 56. 77. 42. 66. 44. 73. 23. 13. 43. 33. 70. 45.
 27. 64. 94. 65. 57. 47. 29. 67. 31. 71. 26. 50. 34. 25. 11. 14. 74. 58.
 15. 60. 39. 38. 76. 55. 22. 21. 20. 28. 19. 62. 61. 63. 10. 53. 54. 49.
 12. 17. 32. 84. 51. 80.]
float64


In [17]:
df['insdutry_groups_id']=df['insdutry_groups_id'].astype(int)

KeyError: 'insdutry_groups_id'

In [23]:
import numpy as np
import pandas as pd

def clean_industry_group(val):
    try:
        val = float(val)
        if np.isnan(val) or val == -1:
            return None
        return int(val)
    except:
        return None

# Apply cleaning function
df['industry_groups_id'] = df['industry_groups_id'].apply(clean_industry_group)

# Convert to pandas nullable Int64 (not numpy int64)
df['industry_groups_id'] = df['industry_groups_id'].astype('Int64')


In [36]:
df['industry_groups_id']

0       81.0
1       81.0
2       81.0
3        NaN
4       81.0
        ... 
5295     NaN
5296    81.0
5297    81.0
5298    81.0
5299    81.0
Name: industry_groups_id, Length: 5300, dtype: float64

In [28]:
df['industry_groups_id'] = df['industry_groups_id'].astype('Int64')  # Ensure nullable int
df['industry_groups_id'] = df['industry_groups_id'].apply(lambda x: '' if pd.isna(x) else x)

df.to_csv("output/companies_cleaned.csv", index=False, encoding='utf-8-sig')

In [34]:
import numpy as np

def clean_industry_group(val):
    try:
        val = float(val)
        if np.isnan(val) or val == -1:
            return None
        return int(val)
    except:
        return None

# Clean the values
df['industry_groups_id'] = df['industry_groups_id'].apply(clean_industry_group)

# Convert to nullable Int64 dtype
df['industry_groups_id'] = df['industry_groups_id'].astype('Int64')

# Save without converting to float
df.to_csv("output/companies_cleaned.csv", index=False, encoding='utf-8-sig')


In [42]:
df['industry_groups_id'] = df['industry_groups_id'].astype('Int64')

In [53]:
df['industry_groups_id'][df['industry_groups_id'].isna()]

Series([], Name: industry_groups_id, dtype: int64)

In [60]:
df.to_csv("output/companies_cleaned.csv", index=False, encoding='utf-8-sig')

In [51]:
import pandas as pd
df=pd.read_csv(r'C:\Users\amirmahdi\Desktop\codal scraper\output\companies_cleaned.csv')

In [41]:
df

,ticker,name,company_id,publisher_status_id,company_type_id,industry_groups_id,company_nature_id,created_at,audited
0,آذرنگین,آذر نگین,811620.0,-1,-1,81.0,1000005,2025-10-29 09:11:47,False
1,بنیاد تعاون نیروی انتظامی جمهوری اسلامی ایران,بنیاد تعاون نیروی انتظامی جمهوری اسلامی ایران,811774.0,-1,-1,81.0,1000005,2025-10-29 09:11:47,False
2,بین المللی خدمات مسافرت هوایی ایرانگردی و جها...,بین المللی خدمات مسافرت هوایی ایرانگردی و جهان...,811602.0,-1,-1,81.0,1000005,2025-10-29 09:11:47,False
3,پترونیک,مهندسی پترونیک صنعت,NaN,-1,-1,NaN,1000005,2025-10-29 09:11:47,False
4,توس پیوند,توس پیوند,811130.0,-1,-1,81.0,1000005,2025-10-29 09:11:47,False
...,...,...,...,...,...,...,...,...,...
5295,کیمیاورزان مهر غرب,کیمیا ورزان مهر غرب,751383.0,4,1,NaN,1000000,2025-10-29 09:11:47,False
5296,کیمیای پارس خاور میانه,کیمیای پارس خاور میانه,812140.0,-1,-1,81.0,1000005,2025-10-29 09:11:47,False
5297,کیمیایی غرب گستر,صنایع کیمیای غرب گستر,811122.0,-1,-1,81.0,1000005,2025-10-29 09:11:47,False
5298,کیهان دریا ایمن قشم,کیهان دریا ایمن قشم,810884.0,-1,-1,81.0,1000000,2025-10-29 09:11:47,False


In [59]:
df['industry_groups_id'][df['industry_groups_id']==81]=-1

C:\Users\amirmahdi\AppData\Local\Temp\ipykernel_6768\1635415117.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['industry_groups_id'][df['industry_groups_id']==81]=-1


In [55]:
# Fetch all existing industry group IDs from SQL
existing_ids = pd.read_sql("SELECT id FROM dbo.industry_groups", con=engine)
existing_ids = set(existing_ids['id'].astype(int))

# Find which IDs in df are not in the SQL table
invalid_ids = set(df['industry_groups_id'].dropna().astype(int)) - existing_ids

print("Invalid industry_groups_id values:", invalid_ids)

NameError: name 'engine' is not defined

In [56]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

# === Configuration ===
csv_path = r"C:\Users\amirmahdi\Desktop\codal scraper\output\companies_cleaned.csv"
server = 'localhost,1433'  # Port must be explicitly specified if using Docker
database = 'Codal_db'
username = 'sa'
password = 'Str0ngP@ssw0rd!'
table_name = 'companies'

# === Build connection string ===
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    f"TrustServerCertificate=yes;"
    f"Encrypt=no;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [57]:
# Fetch all existing industry group IDs from SQL
existing_ids = pd.read_sql("SELECT id FROM dbo.industry_groups", con=engine)
existing_ids = set(existing_ids['id'].astype(int))

# Find which IDs in df are not in the SQL table
invalid_ids = set(df['industry_groups_id'].dropna().astype(int)) - existing_ids

print("Invalid industry_groups_id values:", invalid_ids)

Invalid industry_groups_id values: {81}


In [1]:
import ssl
print(ssl.OPENSSL_VERSION)

OpenSSL 3.0.13 30 Jan 2024


In [1]:
import pandas as pd
df=pd.read_csv(r'C:\Users\amirmahdi\Desktop\codal scraper\output\companies_cleaned.csv')

In [2]:
df

,ticker,name,company_id,publisher_status_id,company_type_id,industry_groups_id,company_nature_id,created_at,audited
0,آذرنگین,آذر نگین,811620.0,-1,-1,-1,1000005,2025-10-29 09:11:47,False
1,بنیاد تعاون نیروی انتظامی جمهوری اسلامی ایران,بنیاد تعاون نیروی انتظامی جمهوری اسلامی ایران,811774.0,-1,-1,-1,1000005,2025-10-29 09:11:47,False
2,بین المللی خدمات مسافرت هوایی ایرانگردی و جها...,بین المللی خدمات مسافرت هوایی ایرانگردی و جهان...,811602.0,-1,-1,-1,1000005,2025-10-29 09:11:47,False
3,پترونیک,مهندسی پترونیک صنعت,NaN,-1,-1,-1,1000005,2025-10-29 09:11:47,False
4,توس پیوند,توس پیوند,811130.0,-1,-1,-1,1000005,2025-10-29 09:11:47,False
...,...,...,...,...,...,...,...,...,...
5295,کیمیاورزان مهر غرب,کیمیا ورزان مهر غرب,751383.0,4,1,-1,1000000,2025-10-29 09:11:47,False
5296,کیمیای پارس خاور میانه,کیمیای پارس خاور میانه,812140.0,-1,-1,-1,1000005,2025-10-29 09:11:47,False
5297,کیمیایی غرب گستر,صنایع کیمیای غرب گستر,811122.0,-1,-1,-1,1000005,2025-10-29 09:11:47,False
5298,کیهان دریا ایمن قشم,کیهان دریا ایمن قشم,810884.0,-1,-1,-1,1000000,2025-10-29 09:11:47,False


In [3]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 17 for SQL Server']


In [1]:
import pandas as pd
df=pd.read_csv('merged_income_statements.csv')

In [2]:
df

,rowCode,label,amount,company_name,source_url,amount as of تاریخ نامشخص
0,49.0,عمليات در حال تداوم:,NaN,قند تربت حيدريه,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
1,3.0,درآمدهاي عملياتي,16420163.0,قند تربت حيدريه,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
2,4.0,بهاى تمام شده درآمدهاي عملياتي,-15258131.0,قند تربت حيدريه,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
3,5.0,سود(زيان) ناخالص,1162032.0,قند تربت حيدريه,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
4,6.0,هزينه‏ هاى فروش، ادارى و عمومى,-423074.0,قند تربت حيدريه,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
...,...,...,...,...,...,...
361,32.0,ناشی از عملیات در حال تداوم,913.0,کشت و صنعت و دامپروري پگاه فارس,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
362,33.0,ناشی از عملیات متوقف شده,0.0,کشت و صنعت و دامپروري پگاه فارس,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
363,34.0,سود (زیان) پایه هر سهم,913.0,کشت و صنعت و دامپروري پگاه فارس,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN
364,35.0,سود (زیان) خالص هر سهم– ریال,913.0,کشت و صنعت و دامپروري پگاه فارس,https://codal.ir/Reports/Decision.aspx?LetterS...,NaN


In [3]:
import pandas as pd
from scraper.label_mapping import label_map


df = pd.DataFrame(list(label_map.items()), columns=["Persian_Label", "English_Key"])

# Step 2: Find duplicate values in 'English_Key' column
duplicates_df = df[df.duplicated(subset="English_Key", keep=False)]

# Step 3: Sort for readability (optional)
duplicates_df = duplicates_df.sort_values(by="English_Key").reset_index(drop=True)

# Show duplicates
duplicates_df

,Persian_Label,English_Key
0,سود (زیان) پایه هر سهم,basic_earnings_loss_per_share
1,سود(زیان) پایه هر سهم,basic_earnings_loss_per_share
2,سود(زیان) پایه هر سهم:,basic_earnings_loss_per_share_colon
3,سود (زیان) پایه هر سهم:,basic_earnings_loss_per_share_colon
4,درآمد سود سهام,dividend_income
5,سود سهام,dividend_income
6,هزینه های مالی,financial_expenses
7,هزینه‏‌هاى مالى,financial_expenses
8,هزینه‏ هاى مالى,financial_expenses
9,سود(زیان) ناخالص,gross_profit_loss


In [5]:
from db.models import IncomeStatement  # adjust path/class name
from sqlalchemy import inspect

# Step 1: Extract columns
columns = IncomeStatement.__table__.columns

# Step 2: Track unique column names
seen = set()
unique_columns = []

for col in columns:
    if col.name not in seen:
        unique_columns.append(col)
        seen.add(col.name)

# Step 3: Rebuild model dynamically (optional step if you want to recreate the class)
with open("columns_list.txt", "w", encoding="utf-8") as f:
    for col in unique_columns:
        f.write(f"{col.name} ({col.type})\n")

In [6]:
from db.base import engine, Base
from db.models import IncomeStatement

IncomeStatement.__table__.create(bind=engine, checkfirst=True)